# AI Leak Detection & Anomaly Recognition (3/4" Domestic Pipe)
### Unsupervised Machine Learning via Isolation Forest & Night-Flow Physics Rules
**SRS Compliance Reference:**
- **FR-05:** Isolation Forest ML algorithm to continuously analyze usage patterns and detect continuous micro-leaks or pipe ruptures.
- **Section 3:** Cloud AI Leak Detection subsystem operating on time-series telemetry.


In [ ]:
# Step 1: Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from datetime import datetime, timedelta
import sqlite3


## Step 2: Hydraulic & Consumption Profile (3/4" Domestic Pipe)
- **Nominal capacity ($Q_n$):** $\approx 25\text{ L/min}$ (tap fully open)
- **Micro-leak threshold:** $\ge 0.15\text{ L/min}$ (leaking flapper or pinhole)
- **Resting hours surveillance:** 01:00 AM – 05:00 AM

In [ ]:
# Step 3: Simulate 14 Days of 10-Minute Domestic Telemetry
np.random.seed(42)
timestamps = []
flow_rates = []

start_time = datetime(2026, 1, 1, 0, 0)
total_intervals = 14 * 24 * 6 # 14 days x 144 intervals/day = 2,016 intervals

for i in range(total_intervals):
    t = start_time + timedelta(minutes=10 * i)
    timestamps.append(t)
    hr = t.hour
    
    # Normal Night (00:00 - 05:00): 99% zero flow
    if 0 <= hr < 5:
        flow = np.random.choice([0.0, 0.0, 0.0, np.random.uniform(1.0, 3.0)])
    # Morning Peak (06:00 - 09:00)
    elif 6 <= hr <= 9:
        flow = np.random.choice([0.0, np.random.uniform(3.0, 15.0), np.random.uniform(1.0, 6.0)])
    # Evening Peak (18:00 - 22:00)
    elif 18 <= hr <= 22:
        flow = np.random.choice([0.0, np.random.uniform(2.0, 12.0), np.random.uniform(0.5, 4.0)])
    # Daytime baseline
    else:
        flow = np.random.choice([0.0, 0.0, np.random.uniform(0.5, 5.0)])
        
    # Inject simulated micro-leak on Day 10 (continuous 0.6 L/min flow between 01:00 - 05:00)
    if 9 <= (i // 144) <= 11 and 1 <= hr < 5:
        flow = np.random.uniform(0.5, 0.8)
        
    flow_rates.append(flow)

df_telemetry = pd.DataFrame({"timestamp": timestamps, "flow_rate_lpm": flow_rates})
df_telemetry["hour"] = df_telemetry["timestamp"].dt.hour
df_telemetry["is_night"] = df_telemetry["hour"].apply(lambda h: 1 if 1 <= h < 5 else 0)
print(f"Generated {len(df_telemetry)} telemetry intervals.")


## Step 4: Fit Isolation Forest Model

In [ ]:
# Feature Matrix: [flow_rate_lpm, hour, is_night]
X = df_telemetry[["flow_rate_lpm", "hour", "is_night"]].values

iso_forest = IsolationForest(n_estimators=100, contamination=0.04, random_state=42)
df_telemetry["anomaly_label"] = iso_forest.fit_predict(X)
df_telemetry["anomaly_score"] = iso_forest.score_samples(X)

anomalies = df_telemetry[df_telemetry["anomaly_label"] == -1]
print(f"Total Anomalies Flagged: {len(anomalies)} ({len(anomalies)/len(df_telemetry)*100:.2f}%)")


## Step 5: Visualize Anomaly Detections on Telemetry Timeline

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(df_telemetry["timestamp"], df_telemetry["flow_rate_lpm"], label="Flow Rate (L/min)", color="#06b6d4", alpha=0.7)
plt.scatter(anomalies["timestamp"], anomalies["flow_rate_lpm"], color="red", label="AI Detected Leak / Anomaly", s=25, zorder=5)
plt.title("14-Day Smart Water Meter Telemetry with AI Leak Detections", fontsize=13, fontweight="bold")
plt.xlabel("Timestamp")
plt.ylabel("Liters / Minute")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
